<!-- SPDX-FileCopyrightText: 2026 Orbital Research Cluster for Celestial Applications (ORCCA) Lab, University of Colorado at Boulder -->
<!-- SPDX-License-Identifier: ISC -->
# N-Plate Model Attitude
---
Last revised by Z. Ellis on 2026 APR 6

## Objectives
This tutorial will demonstrate 

## Imports and Set Up

First we'll import the necessary libraries and load in the supplementary data. Then we define units and frames and load a metakernel to use for time conversions and provide attitude information.

In [1]:
import scarabaeus as scb
import supplementary as supp

import numpy as np
import matplotlib.pyplot as plt

# load tutorial data
data = supp.load_data()

## units, frames, kernels
km, hr = scb.Units.get_units(['km', 'hr'])
J2000 = scb.Frame('J2000')

scb.SpiceManager.clear_kernels()    # ensure clean kernel pool
scb.SpiceManager.load_kernel_from_mkfile(data.OREx_mk.path)

# verify C-Kernels for the S/C (sc) and solar arrays (sa)
scb.SpiceManager.print_kernels()

SCB tutorial data up to date.
                                 Kernels Loaded:
Source:   /Users/zael5647/scarabaeus/docs/online_documentation/sphinx_files/_collections/tutorials/supplementary/data/kernels/scenario/orex_mk.tm   (META)
Source:   data/kernels/scenario/orx_sc_rel_210816_210822_v02.bc   (CK)
Source:   data/kernels/scenario/orx_sa_rel_210816_210822_v02.bc   (CK)
Source:   data/kernels/scenario/ORX_SCLKSCET_00075.tsc   (TEXT)
Source:   data/kernels/locked/lsk/naif0012.tls   (TEXT)
Source:   data/kernels/scenario/orx_v14.tf   (TEXT)


# SAY SOMETHING ABOUT THE DIFFERENT KERNELS LOADED

## Define N-Plate Model & Examine CK Panels
Now that we've loaded in the necessary kernels, we'll need to define an N-Plate model to utilize them. Scarabaeus uses a configuration file, which we loaded in with the rest of the tutorial data:

In [2]:
n_model = scb.nPlateModel(data.OREx_nplate.path)

We'll use `SpiceManager.ckbrief` to look at the SPICE IDs and their matching intervals loaded in with the solar array's attide C-kernel `tutorial_data/kernels/scenario/orx_sa_rel_210816_210822_v02.bc`. Note that we've defined two CK plates in the N-plate configuration JSON, IDs -64017 and -64027 , which are the first and third plate defined in the C-kernel respectively, but the third and fourth plate in the configuration JSON. Since both plates have the same interval, we can just grab the first plate from the brief, corresponding to ID -64207 or Plate 4:

In [3]:
# examine the ID's and intervals within the loaded C-kernel
brief = scb.SpiceManager.ckbrief(data.OREx_ck_sa.path, disp = True)

# N-plate model defines panels -64017 and -64027 -> use their intervals from C-kernel
plate4 = brief[0]    # coresponds to 4th plate in the configuration file
start, end = plate4['TDB_INTERVAL'][0][0], plate4['TDB_INTERVAL'][0][1]     # both plates have same interval

       Brief orx_sa_rel_210816_210822_v02.bc        
ID: -64027
Interval (SCLK): 44718041024704.29 - 44757784483737.375
Interval (TDB): 682343306.1574311 sec (TDB) - 682949743.587035 sec (TDB)

ID: -64022
Interval (SCLK): 44718041024704.29 - 44757784483737.375
Interval (TDB): 682343306.1574311 sec (TDB) - 682949743.587035 sec (TDB)

ID: -64017
Interval (SCLK): 44718041024704.29 - 44757784483737.375
Interval (TDB): 682343306.1574311 sec (TDB) - 682949743.587035 sec (TDB)

ID: -64012
Interval (SCLK): 44718041024704.29 - 44757784483737.375
Interval (TDB): 682343306.1574311 sec (TDB) - 682949743.587035 sec (TDB)



Now we can query the normal vectors for both plates across the entire interval that they've been defined using `nPlateModel.get_ck_normals`:

In [4]:
# get normals information across time interval
orex_id = -64                                         # need s/c ID for SCLK time
dt = scb.ArrayWUnits(0.5, hr)                         # query every half hour
times = scb.EpochArray.interval(start, end, dt)       # the interval to examine

normals = [n_model.get_ck_normals(t, orex_id) for t in times]

TypeError: Equality operation requires both instances to be of type ArrayWUnits.

## Plot C-Kernel Normal Vectors
With the normal vectors queried from the C-kernel, we can plot their time history. This figure will let us scroll through each queried epoch and see how the normal vector changes in the body frame of OSIRIS-REx.

> **NOTE:** the magic command `%matplotlib widget` is only placed in this notebook to allow for the figure to be interactive.

In [5]:
%matplotlib widget
supp.plotting.plot_normals(normals, times)

NameError: name 'normals' is not defined

In [6]:
%matplotlib inline
# NOTE: rerun static version of the plot so that it displays for online tutorial 
supp.plotting.plot_normals(normals, times)

NameError: name 'normals' is not defined

# Conclusion
DESC